In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when

# Impor library MLlib
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier, GBTClassifier
)
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
import pandas as pd

# Hentikan SparkSession jika ada yang aktif
try:
    spark.stop()
except:
    pass

# Buat SparkSession baru
spark = SparkSession.builder \
    .appName("ChurnModeling") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

print("SparkSession dan library MLlib siap.")

SparkSession dan library MLlib siap.


# Load Data & Feature Selection

In [2]:
# 1. Muat data master_feature_table_3.parquet
data_path = "data/master_feature_table_3.parquet"
df = spark.read.parquet(data_path)
df.cache()

# 2. Daftar Fitur yang DIBUANG (Berdasarkan EDA & Korelasi)
cols_to_drop = [
    "msno",                     # ID
    "last_transaction_date",    # Format tanggal
    "last_expiry_date",         # Format tanggal
    
    # --- Berdasarkan Temuan EDA Correlation ---
    # Dibuang karena Redundant (Korelasi > 0.85)
    "count_auto_renew",
    #"total_transactions",       # Korelasi 0.91 dg count_auto_renew
    #"total_payment_plan_days",  # Korelasi 0.88 dg total_transactions
    "total_secs_last_90d",      # Korelasi 0.94 dg total_secs_last_30d
    "active_days_last_90d",     # Korelasi 0.94 dg active_days_last_30d
    
    # Dibuang karena Tidak Prediktif (Berdasarkan EDA)
    "membership_duration_days",
    "registered_via",
    #"city"
    #"lifetime_active_days",
    #"lifetime_unq_songs",
    
]

# 3. Terapkan Feature Selection
df_selected = df.drop(*cols_to_drop)
# df_selected = df_selected.filter(col("age_group") != "Unknown")

print("Feature selection selesai. Skema akhir untuk model:")
df_selected.printSchema()

print("Cek kategori unik di age_group:")
df_selected.select("age_group").distinct().show()

Feature selection selesai. Skema akhir untuk model:
root
 |-- is_churn: integer (nullable = true)
 |-- city: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_payment_plan_days: long (nullable = true)
 |-- avg_discount: double (nullable = true)
 |-- count_cancel: long (nullable = true)
 |-- days_since_last_activity: integer (nullable = true)
 |-- total_secs_last_30d: double (nullable = true)
 |-- active_days_last_30d: long (nullable = true)
 |-- activity_ratio_secs: double (nullable = true)
 |-- percent_complete_last_30d: double (nullable = true)
 |-- lifetime_active_days: long (nullable = true)
 |-- lifetime_unq_songs: long (nullable = true)

Cek kategori unik di age_group:
+------------------+
|         age_group|
+------------------+
|    46-90 (Senior)|
|     0-17 (Remaja)|
|           Unknown|
|    26-35 (Dewasa)|
|      18-25 (Muda)|
|36-45 (Paruh Baya)|
|              NULL|
+------------------+



In [3]:
from pyspark.sql.functions import col, lit

print("--- Distribusi Kelas (Churn vs Not Churn) Setelah Filter ---")

# 1. Hitung Total Baris Data Saat Ini
total_data = df_selected.count()
print(f"Total Data Bersih: {total_data:,} baris")

# 2. Hitung Jumlah per Kelas (0 dan 1)
# Kita group by 'is_churn' lalu hitung count-nya
churn_counts = df_selected.groupBy("is_churn").count().orderBy("is_churn")

# 3. Tambahkan Kolom Persentase
# Rumus: (Jumlah Baris per Kelas / Total Data) * 100
# Kita gunakan fungsi 'withColumn' untuk membuat kolom baru hasil hitungan
churn_distribution = churn_counts.withColumn(
    "percentage", 
    (col("count") / total_data) * 100
)

# 4. Tampilkan Hasil dengan Format Cantik
print("\nTabel Distribusi:")
churn_distribution.show()

# 5. Verifikasi Manual (Opsional agar lebih yakin)
rows = churn_distribution.collect()
# rows[0] adalah baris pertama (biasanya kelas 0), rows[1] adalah kelas 1
count_0 = rows[0]['count']
pct_0 = rows[0]['percentage']

# Cek apakah ada baris kedua (jika data 100% kelas 0, baris kedua error)
if len(rows) > 1:
    count_1 = rows[1]['count']
    pct_1 = rows[1]['percentage']
    
    print(f"Tidak Churn (0): {count_0:,} baris ({pct_0:.2f}%)")
    print(f"Churn (1)      : {count_1:,} baris ({pct_1:.2f}%)")
    print(f"Rasio Imbalance: 1 berbanding {count_0/count_1:.1f}")
else:
    print("Hanya ditemukan satu kelas data!")

--- Distribusi Kelas (Churn vs Not Churn) Setelah Filter ---
Total Data Bersih: 1,082,190 baris

Tabel Distribusi:
+--------+------+-----------------+
|is_churn| count|       percentage|
+--------+------+-----------------+
|       0|983162|90.84929633428511|
|       1| 99028|9.150703665714893|
+--------+------+-----------------+

Tidak Churn (0): 983,162 baris (90.85%)
Churn (1)      : 99,028 baris (9.15%)
Rasio Imbalance: 1 berbanding 9.9


# Define Features Type & Preprocessing Pipeline

In [4]:
# define tipe fitur& pipeline preprocessing
# 1. Tentukan fitur kategorikal dan numerik (dari sisa kolom)
#categorical_cols = ["age_group", "city", "registered_via"]
categorical_cols = ["age_group", "city"]
#categorical_cols = ["age_group"]

# Semua kolom lain selain 'is_churn' dan kategorikal adalah numerik
numerical_cols = [
    col for col in df_selected.columns 
    if col not in categorical_cols + ["is_churn"]
]

print(f"Fitur Kategorikal: {categorical_cols}")
print(f"Fitur Numerik: {numerical_cols}")

# --- TAHAPAN PIPELINE PREPROCESSING ---

# Tahap 1: StringIndexer (Hanya untuk 'age_group' karena 'city' & 'registered_via' sudah angka)
# Kita perlu mengubah "Unknown", "18-25" menjadi 0.0, 1.0, dst.
indexer = StringIndexer(
    inputCol="age_group", 
    outputCol="age_group_idx", 
    handleInvalid="keep" # Mengubah null/unknown menjadi indeks khusus
)

# Tahap 2: OneHotEncoder (Untuk SEMUA kategorikal)
# Mengubah [0.0, 1.0, 2.0] menjadi vector [1,0,0], [0,1,0], [0,0,1]
encoder = OneHotEncoder(
    #inputCols=["age_group_idx", "city", "registered_via"],
    inputCols=["age_group_idx", "city"],
    #outputCols=["age_group_vec", "city_vec", "registered_via_vec"]
    outputCols=["age_group_vec" , "city_vec"]
)

# Tahap 3: VectorAssembler (Hanya untuk fitur NUMERIK)
assembler_num = VectorAssembler(
    inputCols=numerical_cols, 
    outputCol="numerical_features"
)

# Tahap 4: StandardScaler (Untuk fitur numerik)
# Menyamakan skala semua fitur numerik (penting untuk Logistic Regression)
scaler = StandardScaler(
    inputCol="numerical_features", 
    outputCol="scaled_numerical_features"
)

# Tahap 5: VectorAssembler Final (Menggabungkan SEMUA fitur)
assembler_final = VectorAssembler(
    inputCols=[
        "age_group_vec", 
        "city_vec", 
        #"registered_via_vec", 
        "scaled_numerical_features"
    ],
    outputCol="features" # Ini adalah kolom akhir yang dibutuhkan model
)

# Gabungkan semua tahapan preprocessing menjadi satu pipeline
preprocessing_pipeline = Pipeline(
    stages=[
        indexer, 
        encoder, 
        assembler_num, 
        scaler, 
        assembler_final
    ]
)

Fitur Kategorikal: ['age_group', 'city']
Fitur Numerik: ['total_transactions', 'total_payment_plan_days', 'avg_discount', 'count_cancel', 'days_since_last_activity', 'total_secs_last_30d', 'active_days_last_30d', 'activity_ratio_secs', 'percent_complete_last_30d', 'lifetime_active_days', 'lifetime_unq_songs']


In [5]:
jumlah_data = df_selected.count()
print(jumlah_data)

1082190


# Data Splitting & Oversampling/Undersampling (imbalance)

### Tanpa Oversampling/Undersampling

In [6]:
from pyspark.sql.functions import col

# --- 1. PEMBAGIAN DATA ---
print("Membagi data menjadi 80% Latih, 20% Uji...")
train_data, test_data = df_selected.randomSplit([0.8, 0.2], seed=42)

# Cache data latih
train_data.cache()

print(f"Baris data Latih (Original): {train_data.count():,}")

# --- 2. OVERSAMPLING (MENAMBAH 15% DATA MINORITAS) ---
print("\n--- Melakukan Penambahan Data Minoritas (20% dari total) ---")

# Pisahkan kelas
df_majority = train_data.filter(col("is_churn") == 0)
df_minority = train_data.filter(col("is_churn") == 1)

# Hitung jumlah awal
count_maj = df_majority.count()
count_min = df_minority.count()
print(f"Jumlah Awal Mayoritas (0): {count_maj:,}")
print(f"Jumlah Awal Minoritas (1): {count_min:,}")


print(f"Persentase Data Mayoritas: {count_maj/train_data.count()*100:.2f}%")
print(f"Persentase Data Minoritas: {count_min/train_data.count()*100:.2f}%")

Membagi data menjadi 80% Latih, 20% Uji...
Baris data Latih (Original): 865,704

--- Melakukan Penambahan Data Minoritas (20% dari total) ---
Jumlah Awal Mayoritas (0): 786,515
Jumlah Awal Minoritas (1): 79,189
Persentase Data Mayoritas: 90.85%
Persentase Data Minoritas: 9.15%


### Oversampling

In [30]:
from pyspark.sql.functions import col

# --- 1. PEMBAGIAN DATA ---
print("Membagi data menjadi 80% Latih, 20% Uji...")
train_data, test_data = df_selected.randomSplit([0.8, 0.2], seed=42)

# Cache data latih
train_data.cache()

print(f"Baris data Latih (Original): {train_data.count():,}")

# --- 2. OVERSAMPLING (MENAMBAH 15% DATA MINORITAS) ---
print("\n--- Melakukan Penambahan Data Minoritas (16.5% dari total) ---")

# Pisahkan kelas
df_majority = train_data.filter(col("is_churn") == 0)
df_minority = train_data.filter(col("is_churn") == 1)

# Hitung jumlah awal
count_maj = df_majority.count()
count_min = df_minority.count()
print(f"Jumlah Awal Mayoritas (0): {count_maj:,}")
print(f"Jumlah Awal Minoritas (1): {count_min:,}")

# Tentukan rasio tambahan (18%)
ratio_tambahan = (0.165 * train_data.count() - count_min)/count_min
print(f"ratio tambahan: {ratio_tambahan}")

# Ambil sampel tambahan (Hanya mengambil 20% data ekstra)
df_minority_extra = df_minority.sample(
    withReplacement=True, 
    fraction=ratio_tambahan, 
    seed=42
)

# Gabungkan: Mayoritas Asli + Minoritas Asli + Minoritas Ekstra
train_data_oversampled = df_majority.union(df_minority).union(df_minority_extra)

# --- 3. VERIFIKASI ---
print("\n--- Hasil Akhir ---")
print(f"Total Baris Data Latih Baru: {train_data_oversampled.count():,}")

# Cek jumlah per kelas
counts = train_data_oversampled.groupBy("is_churn").count().orderBy("is_churn").collect()
new_count_0 = counts[0]['count']
new_count_1 = counts[1]['count']

print(f"Kelas 0 (Tetap): {new_count_0:,}")
print(f"Kelas 1: {new_count_1:,} (Bertambah {new_count_1 - count_min} baris)")

print(f"Persentase Data Mayoritas: {new_count_0/train_data_oversampled.count()*100:.2f}%")
print(f"Persentase Data Minoritas: {new_count_1/train_data_oversampled.count()*100:.2f}%")

Membagi data menjadi 80% Latih, 20% Uji...
Baris data Latih (Original): 865,704

--- Melakukan Penambahan Data Minoritas (16.5% dari total) ---
Jumlah Awal Mayoritas (0): 786,515
Jumlah Awal Minoritas (1): 79,189
ratio tambahan: 0.8038005278510905

--- Hasil Akhir ---
Total Baris Data Latih Baru: 929,014
Kelas 0 (Tetap): 786,515
Kelas 1: 142,499 (Bertambah 63310 baris)
Persentase Data Mayoritas: 84.66%
Persentase Data Minoritas: 15.34%


In [27]:
train_data_oversampled.printSchema()

root
 |-- is_churn: integer (nullable = true)
 |-- city: integer (nullable = true)
 |-- age_group: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_payment_plan_days: long (nullable = true)
 |-- avg_discount: double (nullable = true)
 |-- count_cancel: long (nullable = true)
 |-- days_since_last_activity: integer (nullable = true)
 |-- total_secs_last_30d: double (nullable = true)
 |-- active_days_last_30d: long (nullable = true)
 |-- activity_ratio_secs: double (nullable = true)
 |-- percent_complete_last_30d: double (nullable = true)
 |-- lifetime_active_days: long (nullable = true)
 |-- lifetime_unq_songs: long (nullable = true)



### Undersampling

In [48]:
# Undersampling
from pyspark.sql.functions import col

# --- 1. PEMBAGIAN DATA ---
print("Membagi data menjadi 80% Latih, 20% Uji...")
train_data, test_data = df_selected.randomSplit([0.8, 0.2], seed=42)

# Cache data latih
train_data.cache()

print(f"Baris data Latih (Original): {train_data.count():,}")

# --- 2. OVERSAMPLING (MENAMBAH 15% DATA MINORITAS) ---
print("\n--- Melakukan Pengurangan Persentase Data Mayoritas Sebanyak 35% dari keseluruhan ---")

# Pisahkan kelas
df_majority = train_data.filter(col("is_churn") == 0)
df_minority = train_data.filter(col("is_churn") == 1)

# Hitung jumlah awal
count_maj = df_majority.count()
count_min = df_minority.count()
print(f"Jumlah Awal Mayoritas (0): {count_maj:,}")
print(f"Jumlah Awal Minoritas (1): {count_min:,}")

# Tentukan rasio kurangin (20%)
ratio_ambil_major = 1-(0.35 * train_data.count() / count_maj)

print(f"ratio_ambil_majority: {ratio_ambil_major}")

# Ambil sampel kurangin (Hanya mengambil 20% data ekstra)
df_majority_sample = df_majority.sample(
    withReplacement=True, 
    fraction=ratio_ambil_major, 
    seed=42
)

# Gabungkan: Mayoritas Asli + Minoritas Asli + Minoritas Ekstra
train_data_oversampled = df_minority.union(df_majority_sample)

# --- 3. VERIFIKASI ---
print("\n--- Hasil Akhir ---")
print(f"Total Baris Data Latih Baru: {train_data_oversampled.count():,}")

# Cek jumlah per kelas
counts = train_data_oversampled.groupBy("is_churn").count().orderBy("is_churn").collect()
new_count_0 = counts[0]['count']
new_count_1 = counts[1]['count']

print(f"Kelas 0: {new_count_0:,} (Berkurang {count_maj - new_count_0} baris)")
print(f"Kelas 1: {new_count_1:,}")

print(f"Persentase Data Mayoritas: {new_count_0/train_data_oversampled.count()*100:.2f}%")
print(f"Persentase Data Minoritas: {new_count_1/train_data_oversampled.count()*100:.2f}%")

Membagi data menjadi 80% Latih, 20% Uji...
Baris data Latih (Original): 865,704

--- Melakukan Pengurangan Persentase Data Mayoritas Sebanyak 35% dari keseluruhan ---
Jumlah Awal Mayoritas (0): 786,515
Jumlah Awal Minoritas (1): 79,189
ratio_ambil_majority: 0.6147608119361996

--- Hasil Akhir ---
Total Baris Data Latih Baru: 561,878
Kelas 0: 482,689 (Berkurang 303826 baris)
Kelas 1: 79,189
Persentase Data Mayoritas: 85.91%
Persentase Data Minoritas: 14.09%


### Undersampling + Oversampling

In [113]:
# Undersampling + Oversampling
from pyspark.sql.functions import col

# --- 1. PEMBAGIAN DATA ---
print("Membagi data menjadi 80% Latih, 20% Uji...")
train_data, test_data = df_selected.randomSplit([0.8, 0.2], seed=42)

# Cache data latih
train_data.cache()

print(f"Baris data Latih (Original): {train_data.count():,}")

# --- 2. OVERSAMPLING (MENAMBAH 15% DATA MINORITAS) ---
print("\n--- Melakukan Pengurangan Data Mayoritas Sebanyak 15% & Penambahan Persentase Data Minoritas Sebanyak 15% dari keseluruhan ---")

# Pisahkan kelas
df_majority = train_data.filter(col("is_churn") == 0)
df_minority = train_data.filter(col("is_churn") == 1)

# Hitung jumlah awal
count_maj = df_majority.count()
count_min = df_minority.count()
print(f"Jumlah Awal Mayoritas (0): {count_maj:,}")
print(f"Jumlah Awal Minoritas (1): {count_min:,}")

# UNDERSAMPLING
# Tentukan rasio kurangin (15%)
ratio_ambil_major = 1-(0.18 * train_data.count() / count_maj)

print(f"ratio_ambil_majority: {ratio_ambil_major}")

# Ambil sampel kurangin (Hanya mengambil 15% data ekstra)
df_majority_sample = df_majority.sample(
    withReplacement=True, 
    fraction=ratio_ambil_major, 
    seed=42
)

# OVERSAMPLING
ratio_tambahan = (0.20 * train_data.count() - count_min)/count_min
print(f"ratio tambahan: {ratio_tambahan}")

df_minority_extra = df_minority.sample(
    withReplacement=True, 
    fraction=ratio_tambahan, 
    seed=42
)

### GABUNGIN UNDERSAMPLING + OVERSAMPLING
train_data_final = df_minority.union(df_majority_sample).union(df_minority_extra)

# --- 3. VERIFIKASI ---
print("\n--- Hasil Akhir ---")
print(f"Total Baris Data Latih Baru: {train_data_final.count():,}")

# Cek jumlah per kelas
counts = train_data_final.groupBy("is_churn").count().orderBy("is_churn").collect()
new_count_0 = counts[0]['count']
new_count_1 = counts[1]['count']

print(f"Kelas 0: {new_count_0:,} (Berkurang {count_maj - new_count_0} baris)")
print(f"Kelas 1: {new_count_1:,}")

print(f"Persentase Data Mayoritas: {new_count_0/train_data_final.count()*100:.2f}%")
print(f"Persentase Data Minoritas: {new_count_1/train_data_final.count()*100:.2f}%")

Membagi data menjadi 80% Latih, 20% Uji...
Baris data Latih (Original): 865,704

--- Melakukan Pengurangan Data Mayoritas Sebanyak 15% & Penambahan Persentase Data Minoritas Sebanyak 15% dari keseluruhan ---
Jumlah Awal Mayoritas (0): 786,515
Jumlah Awal Minoritas (1): 79,189
ratio_ambil_majority: 0.8018769889957598
ratio tambahan: 1.1864248822437462

--- Hasil Akhir ---
Total Baris Data Latih Baru: 802,274
Kelas 0: 629,538 (Berkurang 156977 baris)
Kelas 1: 172,736
Persentase Data Mayoritas: 78.47%
Persentase Data Minoritas: 21.53%


# Define Model & Train Pipeline

In [114]:
# define model & train pipeline
# 1. Definisikan 3 model
lr = LogisticRegression(featuresCol="features", labelCol="is_churn")
rf = RandomForestClassifier(featuresCol="features", labelCol="is_churn", seed=42)
gbt = GBTClassifier(featuresCol="features", labelCol="is_churn", seed=42)

# 2. Buat pipeline lengkap (Preprocessing + Model)
pipeline_lr = Pipeline(stages=[preprocessing_pipeline, lr])
pipeline_rf = Pipeline(stages=[preprocessing_pipeline, rf])
pipeline_gbt = Pipeline(stages=[preprocessing_pipeline, gbt])

# 3. Latih model
# Model dilatih pada data latih yang sudah seimbang (Oversampled)
print("Melatih Logistic Regression...")
model_lr = pipeline_lr.fit(train_data_final)

print("Melatih Random Forest...")
model_rf = pipeline_rf.fit(train_data_final)

print("Melatih GBT...")
model_gbt = pipeline_gbt.fit(train_data_final)

print("Semua model selesai dilatih.")

Melatih Logistic Regression...
Melatih Random Forest...
Melatih GBT...
Semua model selesai dilatih.


In [ ]:
# gbt = model_gbt.stages[-1]
# gbt.featureImportances
# # model_lr.featureImportances
# # model_rf.featureImportances

In [ ]:
# import pandas as pd

# def get_feature_name_mapping(dataset, features_col="features"):
#     """
#     Membongkar metadata kolom features untuk mendapatkan nama asli setiap indeks.
#     """
#     # 1. Ambil Metadata dari skema kolom features
#     meta = dataset.schema[features_col].metadata
    
#     # 2. Cek apakah ada metadata 'ml_attr' (atribut machine learning)
#     if "ml_attr" in meta and "attrs" in meta["ml_attr"]:
#         attrs = meta["ml_attr"]["attrs"]
        
#         # Metadata biasanya terpisah jadi: 'numeric', 'nominal', 'binary'
#         # Kita gabungkan semuanya jadi satu list
#         all_features = []
        
#         if "numeric" in attrs:
#             all_features += attrs["numeric"]
#         if "nominal" in attrs:
#             all_features += attrs["nominal"]
#         if "binary" in attrs:
#             all_features += attrs["binary"]
            
#         # 3. Urutkan berdasarkan indeks (idx)
#         all_features = sorted(all_features, key=lambda x: x['idx'])
        
#         # 4. Buat DataFrame agar mudah dibaca
#         df_map = pd.DataFrame(all_features)
#         return df_map
#     else:
#         print("Metadata ml_attr tidak ditemukan. Pastikan Anda menggunakan DataFrame hasil Transform.")
#         return None

# # --- CARA PAKAI ---

# # Kita butuh DataFrame yang sudah melalui pipeline (karena metadata ada di sana)
# # Kita bisa pakai 'pred_gbt' yang sudah Anda buat sebelumnya
# print("Sedang mengekstrak nama fitur dari metadata...")
# df_feature_names = get_feature_name_mapping(pred_gbt, "features")

# # Tambahkan nilai importance dari GBT Anda
# # Ubah SparseVector menjadi Dictionary agar mudah dimapping
# gbt_imp_dict = {
#     0: 0.0266, 2: 0.0461, 5: 0.0112, 6: 0.0052, 7: 0.0096, 
#     28: 0.2814, 29: 0.1556, 30: 0.0245, 31: 0.0906, 32: 0.1725, 
#     33: 0.0413, 34: 0.0289, 35: 0.0064, 37: 0.069, 38: 0.0311
# }
# # Atau ambil langsung dari model: 
# # gbt_imp_dict = {i: imp for i, imp in enumerate(model_gbt.stages[-1].featureImportances.toArray())}

# # Masukkan ke DataFrame
# df_feature_names['Importance'] = df_feature_names['idx'].map(lambda x: gbt_imp_dict.get(x, 0.0))

# # Tampilkan Hasil yang sudah diurutkan
# print("\n--- PETA FITUR & KEPENTINGANNYA ---")
# print(df_feature_names.sort_values("Importance", ascending=False).head(20).to_string(index=False))

# Model Evaluation on Test Data

In [118]:
# Evaluasi Model pada Data Test (evaluasi balik ke data yg ga seimbang u/ lihat performa nyata model)
# 1. Buat prediksi pada data testing (yang tidak seimbang)
print("Membuat prediksi pada data uji (unseen & imbalanced)...")
pred_lr = model_lr.transform(test_data)
pred_rf = model_rf.transform(test_data)
pred_gbt = model_gbt.transform(test_data)

# 2. Definisikan Evaluator
# menggunakan dua metrik utama untuk data imbalance:
# AUC-ROC: Baik untuk mengukur performa keseluruhan
# AUC-PR: (AreaUnderPrecisionRecall) Sangat baik untuk kelas minoritas yang langka

evaluator_roc = BinaryClassificationEvaluator(
    labelCol="is_churn", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderROC"
)

evaluator_pr = BinaryClassificationEvaluator(
    labelCol="is_churn", 
    rawPredictionCol="rawPrediction", 
    metricName="areaUnderPR"
)

# 3. Hitung dan Tampilkan Hasil
results = {}

print("\n--- Hasil Evaluasi Model ---")

# Logistic Regression
auc_roc_lr = evaluator_roc.evaluate(pred_lr)
auc_pr_lr = evaluator_pr.evaluate(pred_lr)
results['Logistic Regression'] = {'AUC-ROC': auc_roc_lr, 'AUC-PR': auc_pr_lr}
print(f"\nLogistic Regression:")
print(f"  AUC-ROC: {auc_roc_lr:.4f}")
print(f"  AUC-PR (Fokus Churn): {auc_pr_lr:.4f}")

# Random Forest
auc_roc_rf = evaluator_roc.evaluate(pred_rf)
auc_pr_rf = evaluator_pr.evaluate(pred_rf)
results['Random Forest'] = {'AUC-ROC': auc_roc_rf, 'AUC-PR': auc_pr_rf}
print(f"\nRandom Forest:")
print(f"  AUC-ROC: {auc_roc_rf:.4f}")
print(f"  AUC-PR (Fokus Churn): {auc_pr_rf:.4f}")

# GBT
auc_roc_gbt = evaluator_roc.evaluate(pred_gbt)
auc_pr_gbt = evaluator_pr.evaluate(pred_gbt)
results['GBT'] = {'AUC-ROC': auc_roc_gbt, 'AUC-PR': auc_pr_gbt}
print(f"\nGBT Classifier:")
print(f"  AUC-ROC: {auc_roc_gbt:.4f}")
print(f"  AUC-PR (Fokus Churn): {auc_pr_gbt:.4f}")

Membuat prediksi pada data uji (unseen & imbalanced)...

--- Hasil Evaluasi Model ---

Logistic Regression:
  AUC-ROC: 0.9047
  AUC-PR (Fokus Churn): 0.5713

Random Forest:
  AUC-ROC: 0.9116
  AUC-PR (Fokus Churn): 0.6808

GBT Classifier:
  AUC-ROC: 0.9527
  AUC-PR (Fokus Churn): 0.7600


# Confusion Matrix

In [ ]:
# # Confusion Matrix
# from pyspark.mllib.evaluation import MulticlassMetrics

# def print_confusion_matrix(predictions, model_name):
#     # Mengubah prediksi menjadi RDD untuk MulticlassMetrics
#     preds_and_labels = predictions.select("prediction", "is_churn").rdd.map(
#         lambda r: (float(r.prediction), float(r.is_churn))
#     )
    
#     metrics = MulticlassMetrics(preds_and_labels)
#     confusion_matrix = metrics.confusionMatrix().toArray()
    
#     print(f"\n--- Confusion Matrix untuk: {model_name} ---")
#     print(confusion_matrix)
    
#     Overall_Accuracy = metrics.accuracy
#     print(f"  Overall Accuracy:   {Overall_Accuracy:.4f}")

#     # TN, FP
#     # FN, TP
#     TN = confusion_matrix[0][0]
#     FP = confusion_matrix[0][1]
#     FN = confusion_matrix[1][0]
#     TP = confusion_matrix[1][1]
    
#     Recall_Churn = TP / (TP + FN)
#     Precision_Churn = TP / (TP + FP)
#     F1_Churn = 2 * (Precision_Churn * Recall_Churn) / (Precision_Churn + Recall_Churn)
    
#     print(f"  Recall (Churn=1):    {Recall_Churn:.4f}")
#     print(f"  Precision (Churn=1): {Precision_Churn:.4f}")
#     print(f"  F1-Score (Churn=1):  {F1_Churn:.4f}")

# # model terbaik (GBT)
# print_confusion_matrix(pred_gbt, "GBT Classifier")

# # Logistic Regression
# print_confusion_matrix(pred_lr, "Logistic Regression")

# # Random Forest
# print_confusion_matrix(pred_rf, "Random Forest")


In [119]:
from pyspark.sql.functions import col

def evaluate_model_detailed(predictions, model_name):
    """
    Menghitung Confusion Matrix, Accuracy, dan 
    Precision/Recall per kelas (Churn & Not Churn)
    menggunakan DataFrame API yang stabil.
    """
    print(f"\n{'='*40}")
    print(f"DETAIL EVALUASI: {model_name}")
    print(f"{'='*40}")
    
    predictions.cache()
    
    # 1. Hitung Komponen Confusion Matrix
    # TP: Prediksi Churn, Aslinya Churn
    tp = predictions.filter((col("prediction") == 1.0) & (col("is_churn") == 1.0)).count()
    # TN: Prediksi Tidak Churn, Aslinya Tidak Churn
    tn = predictions.filter((col("prediction") == 0.0) & (col("is_churn") == 0.0)).count()
    # FP: Prediksi Churn, Tapi Aslinya Tidak (Salah Alarm)
    fp = predictions.filter((col("prediction") == 1.0) & (col("is_churn") == 0.0)).count()
    # FN: Prediksi Tidak Churn, Tapi Aslinya Churn (Lolos)
    fn = predictions.filter((col("prediction") == 0.0) & (col("is_churn") == 1.0)).count()
    
    total = tp + tn + fp + fn
    
    # 2. Hitung Accuracy Global
    accuracy = (tp + tn) / total if total > 0 else 0.0
    
    # 3. Hitung Metrics untuk Kelas 1 (Churn - Positif)
    # Precision 1: Seberapa tepat saat memprediksi Churn?
    prec_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    # Recall 1: Berapa banyak Churn asli yang tertangkap?
    rec_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    # F1 Score 1
    f1_1 = 2 * (prec_1 * rec_1) / (prec_1 + rec_1) if (prec_1 + rec_1) > 0 else 0.0

    # 4. Hitung Metrics untuk Kelas 0 (Not Churn - Negatif)
    # Precision 0: Seberapa tepat saat memprediksi Tidak Churn?
    # (Pembagi adalah Total Prediksi 0 -> TN + FN)
    prec_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    # Recall 0: Berapa banyak User Setia (0) yang benar terdeteksi?
    # (Pembagi adalah Total Asli 0 -> TN + FP)
    rec_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    # F1 Score 0
    f1_0 = 2 * (prec_0 * rec_0) / (prec_0 + rec_0) if (prec_0 + rec_0) > 0 else 0.0
    
    # --- TAMPILKAN HASIL ---
    
    print(f"Total Data Test: {total}")
    print(f"Accuracy       : {accuracy:.2%}")
    
    print(f"\n[ Confusion Matrix ]")
    print(f"      \t Pred 0 \t Pred 1")
    print(f"Aktual 0: {tn}\t\t {fp}")
    print(f"Aktual 1: {fn}\t\t {tp}")

    print("\n--- Metrik per Kelas ---")
    
    # Tabel Manual Rata Kiri
    print(f"{'KELAS':<15} | {'PRECISION':<10} | {'RECALL':<10} | {'F1-SCORE':<10}")
    print("-" * 55)
    print(f"{'0 (Not Churn)':<15} | {prec_0:.2%}     | {rec_0:.2%}     | {f1_0:.2%}")
    print(f"{'1 (Churn)':<15} | {prec_1:.2%}     | {rec_1:.2%}     | {f1_1:.2%}")
    print("-" * 55)
    
    predictions.unpersist()

evaluate_model_detailed(pred_gbt, "GBT Classifier")
evaluate_model_detailed(pred_rf, "Random Forest")
evaluate_model_detailed(pred_lr, "LR")


DETAIL EVALUASI: GBT Classifier
Total Data Test: 216486
Accuracy       : 93.86%

[ Confusion Matrix ]
      	 Pred 0 	 Pred 1
Aktual 0: 188048		 8599
Aktual 1: 4694		 15145

--- Metrik per Kelas ---
KELAS           | PRECISION  | RECALL     | F1-SCORE  
-------------------------------------------------------
0 (Not Churn)   | 97.56%     | 95.63%     | 96.59%
1 (Churn)       | 63.78%     | 76.34%     | 69.50%
-------------------------------------------------------

DETAIL EVALUASI: Random Forest
Total Data Test: 216486
Accuracy       : 93.96%

[ Confusion Matrix ]
      	 Pred 0 	 Pred 1
Aktual 0: 193136		 3511
Aktual 1: 9570		 10269

--- Metrik per Kelas ---
KELAS           | PRECISION  | RECALL     | F1-SCORE  
-------------------------------------------------------
0 (Not Churn)   | 95.28%     | 98.21%     | 96.72%
1 (Churn)       | 74.52%     | 51.76%     | 61.09%
-------------------------------------------------------

DETAIL EVALUASI: LR
Total Data Test: 216486
Accuracy       : 9

In [14]:
#spark.stop()